## Step 1: CNBC DOW30 -> Get CNBC Profile URLs and Basic Ticker Info

In [29]:
from guidance import system, user, assistant, gen, select
from guidance.models import Transformers
from playwright.async_api import async_playwright
import json
from typing import List, Dict
import re

In [30]:

# Could also do LlamaCpp or many other models
phi_lm = Transformers("Qwen/Qwen2.5-1.5B")

In [31]:
async def fetch_cnbc_page(url: str) -> tuple:
    """
    Fetch CNBC page content using Playwright to handle JS rendering.
    
    Args:
        url: CNBC URL to scrape
        
    Returns:
        Tuple of (html_content, text_content)
    """
    from playwright.async_api import async_playwright
    
    async with async_playwright() as p:
        # Launch browser (headless mode)
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        # Navigate to URL and wait for network to be idle
        await page.goto(url, wait_until="networkidle")
        
        # Wait for any tables or content to load
        try:
            await page.wait_for_selector("table, .cnbc-table, [data-table]", timeout=5000)
        except:
            pass  # Continue even if specific selectors not found
        
        # Get the rendered HTML content
        content = await page.content()
        
        # Also get text content for easier parsing
        text_content = await page.inner_text("body")
        
        await browser.close()
        
        return content, text_content

In [32]:
# Cell 3 - Extract Table Data using Playwright Async (FIXED URL handling)
async def extract_table_data(url: str, debug: bool = True) -> List[Dict[str, str]]:
    """
    Extract table data from CNBC page with company information.
    Uses Playwright to find tables and extract rows.
    
    Args:
        url: CNBC URL containing company table
        debug: Print debug information
        
    Returns:
        List of dictionaries with raw table data
    """
    from playwright.async_api import async_playwright
    
    companies = []
    
    async with async_playwright() as p:
        # Launch browser
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        if debug:
            print(f"Navigating to: {url}")
        
        try:
            # Use domcontentloaded instead of networkidle
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            
            if debug:
                print("Page loaded (domcontentloaded)")
            
            # Wait a bit for JS to execute
            await page.wait_for_timeout(3000)
            
            if debug:
                print("Waited 3 seconds for JS execution")
            
            # Try to wait for tables
            try:
                await page.wait_for_selector("table", timeout=10000)
                if debug:
                    print("Table found!")
            except:
                if debug:
                    print("No table found with wait_for_selector, continuing anyway...")
            
        except Exception as e:
            print(f"Error during page load: {e}")
            await browser.close()
            return []
        
        # Debug: Print page title and URL
        if debug:
            title = await page.title()
            current_url = page.url
            print(f"Page Title: {title}")
            print(f"Current URL: {current_url}")
        
        # Try to find tables
        tables = await page.query_selector_all("table")
        
        if debug:
            print(f"Found {len(tables)} tables")
        
        for table_idx, table in enumerate(tables):
            if debug:
                print(f"\nProcessing table {table_idx + 1}...")
            
            rows = await table.query_selector_all("tr")
            
            if debug:
                print(f"  Found {len(rows)} rows in table {table_idx + 1}")
            
            for row_idx, row in enumerate(rows):
                cells = await row.query_selector_all("td, th")
                
                # Look for company links
                links = await row.query_selector_all("a")
                
                if links and cells:
                    # Extract text from cells
                    cell_texts = []
                    for cell in cells:
                        text = await cell.inner_text()
                        cell_texts.append(text.strip())
                    
                    # Extract links
                    for link in links:
                        href = await link.get_attribute("href")
                        link_text = await link.inner_text()
                        link_text = link_text.strip()
                        
                        if debug and href:
                            print(f"    Row {row_idx}: Link found - Text: '{link_text}', Href: {href}")
                        
                        # Check if it's a company profile link
                        if href and ("/quotes/" in href or "/symbol/" in href or "symbol" in href.lower()):
                            # Fix URL construction to handle different formats
                            if href.startswith("http://") or href.startswith("https://"):
                                # Already a full URL
                                full_url = href
                            elif href.startswith("//"):
                                # Protocol-relative URL (e.g., //www.cnbc.com/quotes/AAPL)
                                full_url = "https:" + href
                            elif href.startswith("/"):
                                # Relative URL (e.g., /quotes/AAPL)
                                full_url = "https://www.cnbc.com" + href
                            else:
                                # Relative path without leading slash
                                full_url = "https://www.cnbc.com/" + href
                            
                            companies.append({
                                "raw_text": " | ".join(cell_texts),
                                "link_text": link_text,
                                "url": full_url,
                                "row_data": cell_texts
                            })
                            
                            if debug:
                                print(f"      ✓ Added company: {link_text} -> {full_url}")
        
        await browser.close()
    
    return companies

In [33]:
def extract_ticker_fallback(text: str) -> str:
    """Fallback regex to extract ticker symbols."""
    # Look for patterns like (AAPL) or uppercase 2-5 letter codes
    match = re.search(r'\(([A-Z]{1,5})\)|\\b([A-Z]{2,5})\\b', text)
    return match.group(1) or match.group(2) if match else ""

In [34]:
def extract_company_info_with_guidance(raw_data: List[Dict], model) -> List[Dict[str, str]]:
    """
    Use guidance to extract structured company information.
    
    Args:
        raw_data: List of raw company data from playwright
        model: Guidance language model
        
    Returns:
        List of structured company dictionaries
    """
    structured_companies = []
    
    for item in raw_data:
        try:
            # Create a prompt for the model to extract structured data
            result = model + f'''
Extract company information from the following data:
Text: {item['raw_text']}
Link Text: {item['link_text']}
URL: {item['url']}

Company Name: {gen('company_name', max_tokens=50, stop='\\n')}
Ticker Symbol: {gen('ticker', max_tokens=10, stop='\\n')}
CNBC Profile URL: {gen('cnbc_url', max_tokens=200, stop='\\n')}
'''
            
            # Extract the generated values
            structured_companies.append({
                "company_name": result['company_name'].strip(),
                "ticker": result['ticker'].strip().upper(),
                "cnbc_url": item['url']  # Use the actual URL we scraped
            })
            
        except Exception as e:
            # Fallback: use regex and basic parsing
            structured_companies.append({
                "company_name": item['link_text'],
                "ticker": extract_ticker_fallback(item['raw_text']),
                "cnbc_url": item['url']
            })
    
    return structured_companies


In [35]:
async def extract_cnbc_companies(cnbc_url: str, use_guidance: bool = True):
    """
    Main pipeline to extract company data from CNBC URL.
    
    Args:
        cnbc_url: The CNBC URL to scrape
        use_guidance: Whether to use guidance for extraction (default: True)
        
    Returns:
        List of company dictionaries with name, ticker, and URL
    """
    print(f"Fetching data from: {cnbc_url}")
    
    # Step 1: Extract raw table data using Playwright
    raw_companies = await extract_table_data(cnbc_url)
    print(f"Found {len(raw_companies)} potential company entries")
    
    # Step 2: Use Guidance for structured extraction (if enabled)
    if use_guidance and raw_companies:
        print("Extracting structured data with guidance...")
        structured_companies = extract_company_info_with_guidance(raw_companies, phi_lm)
    else:
        # Simple extraction without guidance
        structured_companies = []
        for item in raw_companies:
            structured_companies.append({
                "company_name": item['link_text'],
                "ticker": extract_ticker_fallback(item['raw_text']),
                "cnbc_url": item['url']
            })
    
    # Remove duplicates based on URL
    seen_urls = set()
    unique_companies = []
    for company in structured_companies:
        if company['cnbc_url'] not in seen_urls:
            seen_urls.add(company['cnbc_url'])
            unique_companies.append(company)
    
    print(f"Extracted {len(unique_companies)} unique companies")
    return unique_companies

In [36]:
def extract_company_info_direct(raw_data: List[Dict]) -> List[Dict[str, str]]:
    """
    Directly extract structured company information without using LLM.
    This is much faster since the data is already structured from Playwright.
    
    Args:
        raw_data: List of raw company data from playwright
        
    Returns:
        List of structured company dictionaries
    """
    structured_companies = []
    
    for item in raw_data:
        # Extract ticker from URL (e.g., /quotes/AAPL -> AAPL)
        ticker = ""
        if "/quotes/" in item['url']:
            ticker = item['url'].split("/quotes/")[1].split("?")[0].split("/")[0]
        elif "/symbol/" in item['url']:
            ticker = item['url'].split("/symbol/")[1].split("?")[0].split("/")[0]
        
        # If ticker not found in URL, try to extract from raw text
        if not ticker:
            ticker = extract_ticker_fallback(item['raw_text'])
        
        # Clean up company name (remove ticker if it's in parentheses)
        company_name = item['link_text']
        company_name = re.sub(r'\s*\([A-Z]{1,5}\)\s*', '', company_name).strip()
        
        structured_companies.append({
            "ticker": ticker.upper().strip(),
            "company_name": company_name,
            "cnbc_url": item['url']
        })
    
    return structured_companies

In [37]:
async def extract_cnbc_companies(cnbc_url: str) -> List[Dict[str, str]]:
    """
    Main pipeline to extract company data from CNBC URL.
    Fast version without LLM - just direct parsing.
    
    Args:
        cnbc_url: The CNBC URL to scrape
        
    Returns:
        List of company dictionaries with ticker, company_name, and cnbc_url
    """
    print(f"Fetching data from: {cnbc_url}")
    
    # Step 1: Extract raw table data using Playwright
    raw_companies = await extract_table_data(cnbc_url, debug=False)
    print(f"Found {len(raw_companies)} potential company entries")
    
    # Step 2: Direct extraction (NO LLM - much faster!)
    print("Structuring data...")
    structured_companies = extract_company_info_direct(raw_companies)
    
    # Step 3: Remove duplicates based on ticker and URL
    seen_keys = set()
    unique_companies = []
    for company in structured_companies:
        key = (company['ticker'], company['cnbc_url'])
        if key not in seen_keys and company['ticker']:  # Only keep entries with valid tickers
            seen_keys.add(key)
            unique_companies.append(company)
    
    print(f"✓ Extracted {len(unique_companies)} unique companies")
    return unique_companies

In [38]:
async def extract_company_website_from_cnbc(cnbc_url: str, ticker: str, company_name: str, debug: bool = False) -> Dict[str, str]:
    """
    Visit a CNBC company profile page and extract the company's official website/IR page.
    Uses intelligent matching to find URLs containing the company name/ticker.
    
    Args:
        cnbc_url: CNBC company profile URL
        ticker: Company ticker symbol (used to match domain names)
        company_name: Company name (used to match domain names)
        debug: Print debug information
        
    Returns:
        Dictionary with company_website and investor_relations_url
    """
    from playwright.async_api import async_playwright
    import re
    
    result = {
        "company_website": "",
        "investor_relations_url": ""
    }
    
    # Prepare company name variants for matching
    # e.g., "Apple Inc." -> ["apple"]
    # "American Express Company" -> ["americanexpress", "amex"]
    company_words = re.sub(r'[^\w\s]', '', company_name.lower()).split()
    company_words = [w for w in company_words if w not in ['inc', 'corp', 'company', 'corporation', 'ltd', 'llc', 'the', 'group', 'co']]
    ticker_lower = ticker.lower()
    
    # List of third-party sites to ignore
    ignore_domains = [
        'cnbc.com', 'facebook', 'twitter', 'linkedin', 'youtube', 'instagram', 
        'tipranks', 'seekingalpha', 'marketwatch', 'yahoo', 'google', 'reuters',
        'bloomberg', 'morningstar', 'zacks', 'fool.com', 'investopedia'
    ]
    
    def extract_domain(url: str) -> str:
        """Extract the main domain from a URL."""
        match = re.search(r'https?://(?:www\.)?([^/]+)', url)
        return match.group(1).lower() if match else ""
    
    def score_url(url: str, link_text: str = "") -> int:
        """
        Score a URL based on how likely it is to be the company's website.
        Higher score = more likely to be the official site.
        """
        score = 0
        domain = extract_domain(url)
        
        if not domain or any(ignore in domain for ignore in ignore_domains):
            return -1000  # Definitely not the company site
        
        # Check if ticker is in domain
        if ticker_lower in domain:
            score += 100
        
        # Check if company name words are in domain
        for word in company_words:
            if len(word) > 3 and word in domain:  # Only match meaningful words
                score += 50
        
        # Check for IR patterns in URL
        if any(pattern in url.lower() for pattern in ['investor', 'ir.', '/investors', '/investor-relations']):
            score += 30
        
        # Check for IR patterns in link text
        if any(pattern in link_text.lower() for pattern in ['investor', 'ir', 'relations']):
            score += 20
        
        # Prefer cleaner domains (likely official sites)
        if domain.count('.') <= 2:  # e.g., apple.com or investor.apple.com
            score += 10
        
        return score
    
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            page = await browser.new_page()
            
            if debug:
                print(f"  Visiting: {cnbc_url}")
                print(f"  Looking for domains containing: {company_words} or {ticker_lower}")
            
            await page.goto(cnbc_url, wait_until="domcontentloaded", timeout=30000)
            await page.wait_for_timeout(2000)  # Wait for JS
            
            # Collect all external links
            all_links = await page.query_selector_all("a[href^='http']")
            
            candidates = []
            
            for link in all_links:
                href = await link.get_attribute("href")
                if not href:
                    continue
                
                try:
                    link_text = await link.inner_text()
                except:
                    link_text = ""
                
                score = score_url(href, link_text)
                
                if score > 0:  # Only consider positive scores
                    is_ir = any(pattern in href.lower() for pattern in ['investor', 'ir.', '/investors', '/investor-relations'])
                    candidates.append({
                        "url": href,
                        "score": score,
                        "is_ir": is_ir,
                        "text": link_text.strip(),
                        "domain": extract_domain(href)
                    })
                    
                    if debug:
                        print(f"    Candidate: {extract_domain(href)} (score: {score}, IR: {is_ir})")
            
            # Sort by score (highest first)
            candidates.sort(key=lambda x: x['score'], reverse=True)
            
            # Extract best matches
            for candidate in candidates:
                if candidate['is_ir'] and not result["investor_relations_url"]:
                    result["investor_relations_url"] = candidate['url']
                elif not candidate['is_ir'] and not result["company_website"]:
                    result["company_website"] = candidate['url']
                
                # Stop if we found both
                if result["company_website"] and result["investor_relations_url"]:
                    break
            
            # If we only found IR, use it as company website too
            if not result["company_website"] and result["investor_relations_url"]:
                # Try to get the main domain from IR URL
                ir_domain = extract_domain(result["investor_relations_url"])
                # Convert investor.apple.com -> apple.com
                base_domain = '.'.join(ir_domain.split('.')[-2:])
                result["company_website"] = f"https://{base_domain}"
            
            if debug:
                print(f"    ✓ Website: {result['company_website']}")
                print(f"    ✓ IR: {result['investor_relations_url']}")
            
            await browser.close()
            
    except Exception as e:
        if debug:
            print(f"  Error extracting from {cnbc_url}: {e}")
    
    return result


In [39]:
import asyncio

async def enrich_companies_with_websites(companies: List[Dict[str, str]], debug: bool = True) -> List[Dict[str, str]]:
    """
    Enrich company data by visiting each CNBC profile page and extracting
    the company website and investor relations URLs.
    
    Args:
        companies: List of company dictionaries with cnbc_url, ticker, company_name
        debug: Print debug information
        
    Returns:
        Enriched list of company dictionaries
    """
    enriched_companies = []
    
    print(f"\nStep 1.5: Extracting company websites from {len(companies)} CNBC profile pages...")
    print("="*80)
    
    for i, company in enumerate(companies, 1):
        print(f"\n[{i}/{len(companies)}] {company['ticker']} - {company['company_name']}")
        
        # Extract website info from CNBC profile (with company context)
        website_info = await extract_company_website_from_cnbc(
            company['cnbc_url'], 
            company['ticker'],
            company['company_name'],
            debug=debug
        )
        
        # Merge with existing company data
        enriched_company = {
            **company,  # Keep original data
            **website_info  # Add new website data
        }
        
        enriched_companies.append(enriched_company)
        
        # Small delay to be polite to CNBC servers
        if i < len(companies):
            await asyncio.sleep(0.5)  # 500ms delay
    
    print("\n" + "="*80)
    print(f"✓ Enriched {len(enriched_companies)} companies with website information")
    
    return enriched_companies

In [40]:
async def extract_cnbc_companies_full(cnbc_url: str, include_websites: bool = True) -> List[Dict[str, str]]:
    """
    Full pipeline to extract company data from CNBC URL including websites.
    
    Args:
        cnbc_url: The CNBC URL to scrape
        include_websites: Whether to extract company websites from profile pages
        
    Returns:
        List of company dictionaries with ticker, company_name, cnbc_url, and website info
    """
    print("="*80)
    print("STEP 1: Extracting companies from CNBC table")
    print("="*80)
    print(f"Fetching data from: {cnbc_url}")
    
    # Step 1: Extract raw table data using Playwright
    raw_companies = await extract_table_data(cnbc_url, debug=False)
    print(f"Found {len(raw_companies)} potential company entries")
    
    # Step 2: Direct extraction
    print("Structuring data...")
    structured_companies = extract_company_info_direct(raw_companies)
    
    # Remove duplicates
    seen_keys = set()
    unique_companies = []
    for company in structured_companies:
        key = (company['ticker'], company['cnbc_url'])
        if key not in seen_keys and company['ticker']:
            seen_keys.add(key)
            unique_companies.append(company)
    
    print(f"✓ Extracted {len(unique_companies)} unique companies")
    
    # Step 1.5: Extract company websites from CNBC profile pages
    if include_websites:
        unique_companies = await enrich_companies_with_websites(unique_companies, debug=True)
    
    return unique_companies

In [41]:
CNBC_URL = "https://www.cnbc.com/dow-30/"  # Replace with your URL

# Extract companies WITH website information
companies = await extract_cnbc_companies_full(CNBC_URL, include_websites=True)

# Display results
print("\n" + "="*80)
print("FINAL RESULTS - EXTRACTED COMPANIES WITH WEBSITES")
print("="*80)

for i, company in enumerate(companies, 1):
    print(f"\n{i:2d}. {company['ticker']:5s} | {company['company_name']}")
    print(f"     CNBC: {company['cnbc_url']}")
    print(f"     Website: {company.get('company_website', 'N/A')}")
    print(f"     IR: {company.get('investor_relations_url', 'N/A')}")

STEP 1: Extracting companies from CNBC table
Fetching data from: https://www.cnbc.com/dow-30/
Found 30 potential company entries
Structuring data...
✓ Extracted 30 unique companies

Step 1.5: Extracting company websites from 30 CNBC profile pages...

[1/30] AMGN - AMGN
  Visiting: https://www.cnbc.com/quotes/AMGN
  Looking for domains containing: ['amgn'] or amgn
    Candidate: amgen.com (score: 10, IR: False)
    Candidate: apple.news (score: 10, IR: False)
    Candidate: cnbccouncils.com (score: 10, IR: False)
    Candidate: peacocktv.com (score: 10, IR: False)
    Candidate: cnbcrsh.qualtrics.com (score: 10, IR: False)
    Candidate: corporate.comcast.com (score: 10, IR: False)
    Candidate: nbcuniversal.com (score: 10, IR: False)
    Candidate: together.nbcuni.com (score: 10, IR: False)
    Candidate: nbcuniversal.com (score: 10, IR: False)
    Candidate: nbcuniversal.com (score: 10, IR: False)
    Candidate: nbcuniversal.com (score: 10, IR: False)
    Candidate: nbcuniversal.com 

In [42]:
def save_companies_to_json(companies: List[Dict], output_path: str):
    """Save extracted companies to JSON file."""
    with open(output_path, 'w') as f:
        json.dump(companies, f, indent=2)
    print(f"\n✓ Saved {len(companies)} companies to {output_path}")

output_path = "/Users/smatcha/Documents/BigData/investment-report-extractor/data/catalogue/cnbc_companies_full.json"
save_companies_to_json(companies, output_path)


✓ Saved 30 companies to /Users/smatcha/Documents/BigData/investment-report-extractor/data/catalogue/cnbc_companies_full.json


In [43]:
import pandas as pd

df = pd.DataFrame(companies)
print(f"\n{len(df)} companies extracted with full information:\n")
display(df)


30 companies extracted with full information:



,ticker,company_name,cnbc_url,company_website,investor_relations_url
0,AMGN,AMGN,https://www.cnbc.com/quotes/AMGN,https://www.amgen.com/,
1,AMZN,AMZN,https://www.cnbc.com/quotes/AMZN,https://www.amazon.com/,
2,HON,HON,https://www.cnbc.com/quotes/HON,https://www.honeywell.com/us/en,
3,MSFT,MSFT,https://www.cnbc.com/quotes/MSFT,https://www.microsoft.com/en-in/,
4,NVDA,NVDA,https://www.cnbc.com/quotes/NVDA,https://www.nvidia.com/,
5,CSCO,CSCO,https://www.cnbc.com/quotes/CSCO,https://www.cisco.com/,
6,AAPL,AAPL,https://www.cnbc.com/quotes/AAPL,https://www.apple.com/,
7,AXP,AXP,https://www.cnbc.com/quotes/AXP,https://www.americanexpress.com/,
8,BA,BA,https://www.cnbc.com/quotes/BA,https://www.boeing.com,
9,CAT,CAT,https://www.cnbc.com/quotes/CAT,https://www.caterpillar.com,


## Step 2 --> Now use playwright upon each of the company URLs to navigate to investors page and then store it as the value in the investor_relations_url column in the dataframe

In [44]:
# ============================================================================
# STEP 2: Navigate to Company Websites and Find Investor Relations Pages
# ============================================================================

# Cell 9 - Function to find IR links on company website
async def find_investor_relations_page(company_url: str, ticker: str, company_name: str, debug: bool = False) -> str:
    """
    Navigate to a company's website and find their investor relations page.
    Uses multiple strategies including subdomain probing.
    
    Args:
        company_url: The company's main website URL
        ticker: Company ticker (for validation)
        company_name: Company name (for validation)
        debug: Print debug information
        
    Returns:
        The investor relations URL, or empty string if not found
    """
    from playwright.async_api import async_playwright
    import re
    
    if not company_url:
        return ""
    
    # Extract base domain from company URL
    base_domain_match = re.search(r'https?://(?:www\.)?([^/]+)', company_url)
    if not base_domain_match:
        return ""
    
    full_domain = base_domain_match.group(1)
    # Get the root domain (e.g., amazon.com from www.amazon.com)
    domain_parts = full_domain.split('.')
    if len(domain_parts) >= 2:
        root_domain = '.'.join(domain_parts[-2:])  # e.g., amazon.com, honeywell.com
    else:
        root_domain = full_domain
    
    if debug:
        print(f"  Base domain: {full_domain}, Root: {root_domain}")
    
    # STRATEGY 1: Try common IR subdomain patterns first (fastest!)
    common_ir_subdomains = [
        f"https://investor.{root_domain}",
        f"https://investors.{root_domain}",
        f"https://ir.{root_domain}",
        f"https://investorrelations.{root_domain}",
        f"https://s2.q4cdn.com/{ticker.lower()}/",  # Common IR hosting
    ]
    
    # For some companies, IR is on a different domain
    if 'amazon' in root_domain:
        common_ir_subdomains.insert(0, "https://ir.aboutamazon.com")
    elif '3m' in root_domain:
        common_ir_subdomains.insert(0, "https://investors.3m.com")
    elif 'chevron' in root_domain:
        common_ir_subdomains.insert(0, "https://www.chevron.com/investors")
    elif 'caterpillar' in root_domain:
        common_ir_subdomains.insert(0, "https://www.caterpillar.com/en/investors.html")
    elif 'salesforce' in root_domain:
        common_ir_subdomains.insert(0, "https://investor.salesforce.com")
    elif 'pg.com' in root_domain or 'proctergamble' in root_domain:
        common_ir_subdomains.insert(0, "https://www.pginvestor.com")
    elif 'visa' in root_domain:
        common_ir_subdomains.insert(0, "https://investor.visa.com")
    
    if debug:
        print(f"  Trying common IR subdomain patterns...")
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        
        # Try each common subdomain
        for subdomain_url in common_ir_subdomains:
            if debug:
                print(f"    Testing: {subdomain_url}")
            
            try:
                page = await browser.new_page()
                response = await page.goto(subdomain_url, wait_until="domcontentloaded", timeout=15000)
                
                # Check if page loaded successfully (200 status)
                if response and response.status == 200:
                    # Verify it's actually an IR page by checking content
                    await page.wait_for_timeout(1000)
                    content = await page.content()
                    content_lower = content.lower()
                    
                    # Check for IR-related keywords in content
                    ir_keywords = ['investor relations', 'investor', 'sec filings', 'financial results', 'earnings', 'stock information']
                    keyword_count = sum(1 for keyword in ir_keywords if keyword in content_lower)
                    
                    if keyword_count >= 2:  # At least 2 IR keywords found
                        if debug:
                            print(f"    ✓ Found valid IR page: {subdomain_url}")
                        await page.close()
                        await browser.close()
                        return subdomain_url
                
                await page.close()
            except Exception as e:
                if debug:
                    print(f"    ✗ Failed: {e}")
                continue
        
        # STRATEGY 2: Scrape the main website for IR links
        if debug:
            print(f"  Subdomain probing failed, scraping main website...")
        
        try:
            page = await browser.new_page()
            await page.goto(company_url, wait_until="domcontentloaded", timeout=30000)
            await page.wait_for_timeout(2000)
            
            if debug:
                print(f"  Page loaded: {await page.title()}")
            
            # Look for ALL links (including footer, header, navigation)
            all_links = await page.query_selector_all("a[href]")
            
            candidates = []
            
            for link in all_links:
                try:
                    href = await link.get_attribute("href")
                    if not href:
                        continue
                    
                    # Get link text and surrounding context
                    link_text = await link.inner_text()
                    link_text_lower = link_text.strip().lower()
                    
                    # Also check aria-label and title attributes
                    aria_label = await link.get_attribute("aria-label")
                    title_attr = await link.get_attribute("title")
                    
                    combined_text = f"{link_text_lower} {aria_label or ''} {title_attr or ''}".lower()
                    
                    # Convert relative URLs to absolute
                    if href.startswith('/'):
                        href = company_url.rstrip('/') + href
                    elif href.startswith('http'):
                        pass  # Already absolute
                    else:
                        href = company_url.rstrip('/') + '/' + href
                    
                    href_lower = href.lower()
                    
                    # Score the link
                    score = 0
                    
                    # URL patterns (high value)
                    url_patterns = {
                        'investor.': 50,
                        'investors.': 50,
                        'ir.': 50,
                        '/investor': 40,
                        '/investors': 40,
                        '/investor-relations': 45,
                        '/investorrelations': 45,
                        '/shareholder': 35,
                        '/ir': 30,
                    }
                    
                    for pattern, points in url_patterns.items():
                        if pattern in href_lower:
                            score += points
                    
                    # Text patterns (medium value)
                    text_patterns = {
                        'investor relations': 40,
                        'investors': 30,
                        'investor': 25,
                        'shareholder': 20,
                        'financial information': 20,
                        'stock information': 20,
                        'ir': 15,
                    }
                    
                    for pattern, points in text_patterns.items():
                        if pattern in combined_text:
                            score += points
                    
                    # Only keep links related to the company domain OR known IR subdomains
                    if score > 0:
                        # Check if it's on the same root domain or a known IR subdomain
                        is_valid_domain = (
                            root_domain in href_lower or
                            any(sub in href_lower for sub in ['investor.', 'investors.', 'ir.', 'q4cdn.com'])
                        )
                        
                        if is_valid_domain:
                            candidates.append({
                                'url': href,
                                'text': link_text.strip(),
                                'score': score
                            })
                            
                            if debug:
                                print(f"    Candidate: {link_text.strip()[:40]} -> {href} (score: {score})")
                
                except Exception as e:
                    continue
            
            await page.close()
            await browser.close()
            
            # Return the highest scoring candidate
            if candidates:
                candidates.sort(key=lambda x: x['score'], reverse=True)
                best_match = candidates[0]
                
                if debug:
                    print(f"  ✓ Best match: {best_match['url']}")
                
                return best_match['url']
            
            if debug:
                print(f"  ✗ No investor relations link found")
            
            return ""
        
        except Exception as e:
            if debug:
                print(f"  Error scraping main website: {e}")
            await browser.close()
            return ""


In [45]:

# Cell 10 - Function to process all companies and find IR pages
async def find_all_ir_pages(companies: List[Dict[str, str]], debug: bool = True) -> List[Dict[str, str]]:
    """
    Process all companies and find their investor relations pages.
    
    Args:
        companies: List of company dictionaries with company_website field
        debug: Print debug information
        
    Returns:
        Updated list with investor_relations_url filled in
    """
    updated_companies = []
    
    print("\n" + "="*80)
    print("STEP 2: Finding Investor Relations Pages on Company Websites")
    print("="*80)
    
    for i, company in enumerate(companies, 1):
        print(f"\n[{i}/{len(companies)}] {company['ticker']} - {company['company_name']}")
        
        # Use existing IR URL if already found in Step 1, otherwise search
        if company.get('investor_relations_url') and company['investor_relations_url']:
            print(f"  ✓ Already have IR URL: {company['investor_relations_url']}")
            updated_companies.append(company)
        else:
            # Navigate to company website and find IR page
            company_website = company.get('company_website', '')
            
            if not company_website:
                print(f"  ✗ No company website found, skipping...")
                updated_companies.append(company)
                continue
            
            ir_url = await find_investor_relations_page(
                company_website,
                company['ticker'],
                company['company_name'],
                debug=debug
            )
            
            # Update company with IR URL
            updated_company = {**company}
            if ir_url:
                updated_company['investor_relations_url'] = ir_url
                print(f"  ✓ Found IR page: {ir_url}")
            else:
                # Keep company website as fallback
                if not updated_company.get('investor_relations_url'):
                    updated_company['investor_relations_url'] = company_website
                print(f"  ⚠ Using company website as fallback")
            
            updated_companies.append(updated_company)
        
        # Be polite to servers
        if i < len(companies):
            await asyncio.sleep(1.0)  # 1 second delay between requests
    
    print("\n" + "="*80)
    print(f"✓ Processed {len(updated_companies)} companies")
    
    # Count how many have IR URLs
    with_ir = sum(1 for c in updated_companies if c.get('investor_relations_url'))
    print(f"✓ {with_ir}/{len(updated_companies)} companies have investor relations URLs")
    print("="*80)
    
    return updated_companies


In [46]:

# Cell 11 - Execute Step 2: Find IR Pages
# Assuming 'companies' dataframe/list exists from Step 1
print("Starting Step 2: Finding Investor Relations Pages...")

# Run Step 2
companies_with_ir = await find_all_ir_pages(companies, debug=True)

# Update the companies variable
companies = companies_with_ir


Starting Step 2: Finding Investor Relations Pages...

STEP 2: Finding Investor Relations Pages on Company Websites

[1/30] AMGN - AMGN
  Base domain: amgen.com, Root: amgen.com
  Trying common IR subdomain patterns...
    Testing: https://investor.amgen.com
    ✗ Failed: Page.goto: net::ERR_NAME_NOT_RESOLVED at https://investor.amgen.com/
Call log:
  - navigating to "https://investor.amgen.com/", waiting until "domcontentloaded"

    Testing: https://investors.amgen.com
    Testing: https://ir.amgen.com
    ✗ Failed: Page.goto: net::ERR_NAME_NOT_RESOLVED at https://ir.amgen.com/
Call log:
  - navigating to "https://ir.amgen.com/", waiting until "domcontentloaded"

    Testing: https://investorrelations.amgen.com
    ✗ Failed: Page.goto: net::ERR_NAME_NOT_RESOLVED at https://investorrelations.amgen.com/
Call log:
  - navigating to "https://investorrelations.amgen.com/", waiting until "domcontentloaded"

    Testing: https://s2.q4cdn.com/amgn/
  Subdomain probing failed, scraping main we

In [47]:

# Cell 12 - Display Results
print("\n" + "="*80)
print("STEP 2 RESULTS - Companies with IR Pages")
print("="*80)

for i, company in enumerate(companies, 1):
    print(f"\n{i:2d}. {company['ticker']:5s} | {company['company_name']}")
    print(f"     Website: {company.get('company_website', 'N/A')}")
    print(f"     IR Page: {company.get('investor_relations_url', 'N/A')}")



STEP 2 RESULTS - Companies with IR Pages

 1. AMGN  | AMGN
     Website: https://www.amgen.com/
     IR Page: http://investors.amgen.com/

 2. AMZN  | AMZN
     Website: https://www.amazon.com/
     IR Page: https://ir.aboutamazon.com

 3. HON   | HON
     Website: https://www.honeywell.com/us/en
     IR Page: https://www.honeywell.com/us/en

 4. MSFT  | MSFT
     Website: https://www.microsoft.com/en-in/
     IR Page: https://www.microsoft.com/investor/default.aspx

 5. NVDA  | NVDA
     Website: https://www.nvidia.com/
     IR Page: https://investor.nvidia.com

 6. CSCO  | CSCO
     Website: https://www.cisco.com/
     IR Page: https://investor.cisco.com

 7. AAPL  | AAPL
     Website: https://www.apple.com/
     IR Page: https://investor.apple.com

 8. AXP   | AXP
     Website: https://www.americanexpress.com/
     IR Page: https://ir.americanexpress.com

 9. BA    | BA
     Website: https://www.boeing.com
     IR Page: https://investors.boeing.com

10. CAT   | CAT
     Website: ht

In [48]:

# Cell 13 - Save Updated Data
output_path = "/Users/smatcha/Documents/BigData/investment-report-extractor/data/catalogue/cnbc_companies_with_ir.json"
save_companies_to_json(companies, output_path)

# Also save to CSV
import pandas as pd
df = pd.DataFrame(companies)

csv_path = "/Users/smatcha/Documents/BigData/investment-report-extractor/data/catalogue/cnbc_companies_with_ir.csv"
df.to_csv(csv_path, index=False)
print(f"✓ Saved to CSV: {csv_path}")

# Display DataFrame
print(f"\n{len(df)} companies with complete information:\n")
display(df)



✓ Saved 30 companies to /Users/smatcha/Documents/BigData/investment-report-extractor/data/catalogue/cnbc_companies_with_ir.json
✓ Saved to CSV: /Users/smatcha/Documents/BigData/investment-report-extractor/data/catalogue/cnbc_companies_with_ir.csv

30 companies with complete information:



,ticker,company_name,cnbc_url,company_website,investor_relations_url
0,AMGN,AMGN,https://www.cnbc.com/quotes/AMGN,https://www.amgen.com/,http://investors.amgen.com/
1,AMZN,AMZN,https://www.cnbc.com/quotes/AMZN,https://www.amazon.com/,https://ir.aboutamazon.com
2,HON,HON,https://www.cnbc.com/quotes/HON,https://www.honeywell.com/us/en,https://www.honeywell.com/us/en
3,MSFT,MSFT,https://www.cnbc.com/quotes/MSFT,https://www.microsoft.com/en-in/,https://www.microsoft.com/investor/default.aspx
4,NVDA,NVDA,https://www.cnbc.com/quotes/NVDA,https://www.nvidia.com/,https://investor.nvidia.com
5,CSCO,CSCO,https://www.cnbc.com/quotes/CSCO,https://www.cisco.com/,https://investor.cisco.com
6,AAPL,AAPL,https://www.cnbc.com/quotes/AAPL,https://www.apple.com/,https://investor.apple.com
7,AXP,AXP,https://www.cnbc.com/quotes/AXP,https://www.americanexpress.com/,https://ir.americanexpress.com
8,BA,BA,https://www.cnbc.com/quotes/BA,https://www.boeing.com,https://investors.boeing.com
9,CAT,CAT,https://www.cnbc.com/quotes/CAT,https://www.caterpillar.com,https://investors.caterpillar.com


In [49]:

# Cell 14 - Quality Check: Show companies without IR pages
companies_without_ir = [c for c in companies if not c.get('investor_relations_url') or c.get('investor_relations_url') == c.get('company_website')]

if companies_without_ir:
    print("\n" + "="*80)
    print(f"⚠ {len(companies_without_ir)} companies may need manual verification:")
    print("="*80)
    for company in companies_without_ir:
        print(f"  - {company['ticker']:5s} | {company['company_name']}")
        print(f"    Website: {company.get('company_website', 'N/A')}")


⚠ 2 companies may need manual verification:
  - HON   | HON
    Website: https://www.honeywell.com/us/en
  - CVX   | CVX
    Website: https://www.chevron.com/


## Step 3 --> Extract All the valuable information that is available, that is in the IR page, in a structured format and provide the structured output using guidance 